### Model Design Challenge (Extra credit: Maximum 20 points) 🚀

In [8]:
# Setup: data, dataloaders, baseline/challenge models, and training helpers.
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset
import yfinance as yf

SEED = 42
pl.seed_everything(SEED, workers=True)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

ARTIFACT_DIR = Path("challenge_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_STOCKS = [
    "TSLA",
    "META",
    "NVDA",
    "AMZN",
    "NFLX",
    "GBTC",
    "GDX",
    "INTC",
    "DAL",
    "C",
    "GOOG",
    "AAPL",
    "MSFT",
    "IBM",
    "HPQ",
    "ORCL",
    "SAP",
    "CRM",
    "HUBS",
    "TWLO",
]
TARGET_STOCK = "MSFT"
START_DATE = "2020-01-01"
DAYS = 24  # <= 32 per challenge rule.


def download_close_prices(tickers, start_date):
    data = yf.download(tickers=tickers, start=start_date, auto_adjust=True, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data = data["Close"]
    if isinstance(data, pd.Series):
        data = data.to_frame()
    return data.dropna()


def build_windows(feature_frame, target_series, days):
    feature_np = feature_frame.values.astype(np.float32).T  # (n_stocks, n_days_total)
    target_np = target_series.values.astype(np.float32)

    xs, ys = [], []
    for i in range(len(target_np) - days):
        xs.append(feature_np[:, i:i + days])
        ys.append(target_np[i + days])

    return np.stack(xs), np.asarray(ys, dtype=np.float32)


prices = download_close_prices(FEATURE_STOCKS + [TARGET_STOCK], START_DATE)
feature_df = prices[FEATURE_STOCKS].copy()
target_series = prices[TARGET_STOCK].copy()

X, y = build_windows(feature_df, target_series, DAYS)

n_total = len(X)
perm = np.random.default_rng(SEED).permutation(n_total)

n_train = int(0.70 * n_total)
n_valid = int(0.15 * n_total)
n_test = n_total - n_train - n_valid

train_idx = perm[:n_train]
valid_idx = perm[n_train:n_train + n_valid]
test_idx = perm[n_train + n_valid:]

X_train, y_train = X[train_idx], y[train_idx]
X_valid, y_valid = X[valid_idx], y[valid_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Standardize using train-only statistics.
x_mean = X_train.mean(axis=(0, 2), keepdims=True)
x_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
y_mean = y_train.mean()
y_std = y_train.std() + 1e-6

X_train_scaled = (X_train - x_mean) / x_std
X_valid_scaled = (X_valid - x_mean) / x_std
X_test_scaled = (X_test - x_mean) / x_std

y_train_scaled = (y_train - y_mean) / y_std
y_valid_scaled = (y_valid - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std


class WindowDataset(Dataset):

    def __init__(self, x_np, y_np):
        self.x = torch.tensor(x_np, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


train_dataset = WindowDataset(X_train_scaled, y_train_scaled)
valid_dataset = WindowDataset(X_valid_scaled, y_valid_scaled)
test_dataset = WindowDataset(X_test_scaled, y_test_scaled)

batch_size = min(128, len(train_dataset))

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
train_eval_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

Y_MEAN = float(y_mean)
Y_STD = float(y_std)


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class PreviouslyConstructedCNN(nn.Module):
    """Baseline from Section 2 style: Conv1d + Dropout."""

    def __init__(self, in_channels=20, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x).squeeze(1)


class DepthwiseSeparableBlock(nn.Module):

    def __init__(self, in_channels, out_channels, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.block(x)


class CompactChallengeCNN(nn.Module):
    """Challenge model: fewer params than baseline but still Conv1d + Dropout."""

    def __init__(self, in_channels=20, dropout=0.25):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.block1 = DepthwiseSeparableBlock(32, 48, dropout=dropout)
        self.block2 = DepthwiseSeparableBlock(48, 64, dropout=dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x)
        return self.head(x).squeeze(1)


class LitRegressor(pl.LightningModule):

    def __init__(self, backbone, lr=1e-3, weight_decay=1e-6):
        super().__init__()
        self.save_hyperparameters(ignore=["backbone"])
        self.backbone = backbone
        self.lr = lr
        self.weight_decay = weight_decay
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.backbone(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log("train_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log("val_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log("test_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)


@torch.no_grad()
def raw_mse(model, dataloader, y_mean_scalar, y_std_scalar):
    model.eval()
    device = model.device
    y_mean_t = torch.tensor(y_mean_scalar, dtype=torch.float32, device=device)
    y_std_t = torch.tensor(y_std_scalar, dtype=torch.float32, device=device)

    sq_err_sum = 0.0
    n = 0
    for xb, yb in dataloader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred_scaled = model(xb)

        pred_raw = pred_scaled * y_std_t + y_mean_t
        true_raw = yb * y_std_t + y_mean_t

        sq_err_sum += torch.sum((pred_raw - true_raw)**2).item()
        n += yb.numel()

    return sq_err_sum / max(n, 1)


def train_one_model(backbone, run_name, max_epochs=80, lr=1e-3, weight_decay=1e-6, patience=12):
    model = LitRegressor(backbone=backbone, lr=lr, weight_decay=weight_decay)

    run_logger = CSVLogger(save_dir=str(ARTIFACT_DIR), name=run_name)
    ckpt_callback = ModelCheckpoint(
        monitor="val_mse",
        mode="min",
        save_top_k=1,
        filename="best-{epoch:02d}-{val_mse:.4f}",
    )
    early_stop = EarlyStopping(monitor="val_mse", mode="min", patience=patience)

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        logger=run_logger,
        callbacks=[ckpt_callback, early_stop],
        deterministic=True,
        log_every_n_steps=1,
        enable_model_summary=False,
    )

    trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=valid_dataloader)

    if ckpt_callback.best_model_path:
        ckpt = torch.load(ckpt_callback.best_model_path, map_location="cpu")
        model.load_state_dict(ckpt["state_dict"])

    train_mse_raw = raw_mse(model, train_eval_dataloader, Y_MEAN, Y_STD)
    val_mse_raw = raw_mse(model, valid_dataloader, Y_MEAN, Y_STD)
    test_mse_raw = raw_mse(model, test_dataloader, Y_MEAN, Y_STD)

    summary = {
        "run_name": run_name,
        "log_dir": run_logger.log_dir,
        "best_ckpt_path": ckpt_callback.best_model_path,
        "train_mse_raw": train_mse_raw,
        "val_mse_raw": val_mse_raw,
        "test_mse_raw": test_mse_raw,
    }

    return model, summary


print(f"Dataset windows: {n_total}")
print(f"Split sizes  -> train: {n_train}, val: {n_valid}, test: {n_test}")
print(f"Input tensor -> {X.shape} (samples, stocks, days)")
print(f"Challenge window (days): {DAYS}")


Seed set to 42


Dataset windows: 1526
Split sizes  -> train: 1068, val: 228, test: 230
Input tensor -> (1526, 20, 24) (samples, stocks, days)
Challenge window (days): 24


In [9]:
# Train baseline and challenge models, then compare parameter counts and MSE.
baseline_backbone = PreviouslyConstructedCNN(in_channels=len(FEATURE_STOCKS), dropout=0.30)
challenge_backbone = CompactChallengeCNN(in_channels=len(FEATURE_STOCKS), dropout=0.25)

baseline_params = count_trainable_params(baseline_backbone)
challenge_params = count_trainable_params(challenge_backbone)

print(f"Baseline parameter count:  {baseline_params:,}")
print(f"Challenge parameter count: {challenge_params:,}")

if challenge_params > baseline_params:
    raise ValueError("Challenge model exceeds parameter budget.")

baseline_model, baseline_summary = train_one_model(
    backbone=baseline_backbone,
    run_name="baseline_prev_cnn",
    max_epochs=60,
    lr=1e-3,
    weight_decay=1e-6,
    patience=10,
)

challenge_model, challenge_summary = train_one_model(
    backbone=challenge_backbone,
    run_name="challenge_compact_cnn",
    max_epochs=80,
    lr=8e-4,
    weight_decay=1e-5,
    patience=12,
)

print("\nRaw-price MSE (lower is better):")
print(f"Baseline  -> train: {baseline_summary['train_mse_raw']:.4f}, "
      f"val: {baseline_summary['val_mse_raw']:.4f}, "
      f"test: {baseline_summary['test_mse_raw']:.4f}")
print(f"Challenge -> train: {challenge_summary['train_mse_raw']:.4f}, "
      f"val: {challenge_summary['val_mse_raw']:.4f}, "
      f"test: {challenge_summary['test_mse_raw']:.4f}")

if challenge_summary["test_mse_raw"] < 120:
    print("Challenge score target met: test MSE < 120.")
if challenge_summary["test_mse_raw"] < 20:
    print("Challenge top threshold met: test MSE < 20.")

results = {
    "days": DAYS,
    "feature_stocks": FEATURE_STOCKS,
    "target_stock": TARGET_STOCK,
    "baseline_params": baseline_params,
    "challenge_params": challenge_params,
    "parameter_budget_ok": challenge_params <= baseline_params,
    "baseline": baseline_summary,
    "challenge": challenge_summary,
}

results_path = ARTIFACT_DIR / "challenge_results.json"
results_path.write_text(json.dumps(results, indent=2))
print(f"\nSaved summary JSON: {results_path.resolve()}")

for name, summary in [("baseline", baseline_summary), ("challenge", challenge_summary)]:
    metrics_path = Path(summary["log_dir"]) / "metrics.csv"
    if not metrics_path.exists():
        continue

    metrics = pd.read_csv(metrics_path)
    keep_cols = [c for c in ["epoch", "step", "train_mse", "train_mse_epoch", "val_mse", "test_mse"] if c in metrics.columns]

    print(f"\n{name} log file: {metrics_path}")
    print(metrics[keep_cols].tail(12).to_string(index=False))


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Baseline parameter count:  10,273
Challenge parameter count: 7,089


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Raw-price MSE (lower is better):
Baseline  -> train: 125.7870, val: 121.6170, test: 103.4796
Challenge -> train: 115.3182, val: 113.6468, test: 88.6939
Challenge score target met: test MSE < 120.

Saved summary JSON: /Users/shouyueliu/projects/ann_2026/challenge_artifacts/challenge_results.json

baseline log file: challenge_artifacts/baseline_prev_cnn/version_1/metrics.csv
 epoch  step  train_mse  val_mse
    54   494        NaN 0.017404
    54   494   0.055861      NaN
    55   503        NaN 0.011734
    55   503   0.056381      NaN
    56   512        NaN 0.016015
    56   512   0.052728      NaN
    57   521        NaN 0.013663
    57   521   0.063527      NaN
    58   530        NaN 0.012818
    58   530   0.056428      NaN
    59   539        NaN 0.013274
    59   539   0.054147      NaN

challenge log file: challenge_artifacts/challenge_compact_cnn/version_2/metrics.csv
 epoch  step  train_mse  val_mse
    42   386        NaN 0.017500
    42   386   0.028744      NaN
    43   3

### Upload model to Hugging Face Model Hub
Run the next cell after training. It saves artifacts locally and uploads them to your HF model repo.

In [10]:
# Save artifacts locally and upload to Hugging Face Model Hub.
import shutil

if "challenge_model" not in globals() or "challenge_summary" not in globals():
    raise RuntimeError("Run the training cell above first so `challenge_model` and summaries exist.")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Save compact challenge model weights.
weights_path = ARTIFACT_DIR / "compact_challenge_cnn_state_dict.pt"
torch.save(challenge_model.backbone.state_dict(), weights_path)

# Save full Lightning state_dict for reproducibility.
full_state_path = ARTIFACT_DIR / "compact_challenge_litregressor_state_dict.pt"
torch.save(challenge_model.state_dict(), full_state_path)

# Persist a concise model card + metadata.
model_card = f"""---
license: mit
library_name: pytorch
tags:
- time-series-forecasting
- pytorch-lightning
- ann2026
---

# ANN-2026 HW2 Challenge Model

## Task
Forecast MSFT close price from a 24-day window of 20 related stock time series.

## Parameter Budget
- Baseline params: {baseline_params:,}
- Challenge params: {challenge_params:,}
- Budget satisfied: {challenge_params <= baseline_params}

## Final Metrics (raw-price MSE)
- Baseline test MSE: {baseline_summary['test_mse_raw']:.4f}
- Challenge test MSE: {challenge_summary['test_mse_raw']:.4f}

## Notes
Training logs are included under `logs/` and full run metadata is in `challenge_results.json`.
"""
(ARTIFACT_DIR / "README.md").write_text(model_card)

# Copy logger directories so the notebook + logs can be shared on GitHub.
logs_dir = ARTIFACT_DIR / "logs"
logs_dir.mkdir(exist_ok=True)
for name, summary in [("baseline", baseline_summary), ("challenge", challenge_summary)]:
    src = Path(summary["log_dir"])
    dst = logs_dir / name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)

print(f"Saved artifacts to: {ARTIFACT_DIR.resolve()}")
print("Commit `hw2.challenge.ipynb` and `challenge_artifacts/` to GitHub for submission.")

# Hugging Face upload.
from huggingface_hub import HfApi, create_repo

repo_id = 'iaaronlau/ANN2026-HW2-Challenge'
if repo_id:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    api = HfApi()
    api.upload_folder(
        repo_id=repo_id,
        repo_type="model",
        folder_path=str(ARTIFACT_DIR),
        commit_message="Upload ANN-2026 HW2 challenge model + logs",
    )
    print(f"Uploaded to: https://huggingface.co/{repo_id}")
else:
    print("HF upload skipped. Set env var `HF_REPO_ID` (e.g., 'username/ann2026-hw2-challenge') and rerun.")


Saved artifacts to: /Users/shouyueliu/projects/ann_2026/challenge_artifacts
Commit `hw2.challenge.ipynb` and `challenge_artifacts/` to GitHub for submission.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to: https://huggingface.co/iaaronlau/ANN2026-HW2-Challenge
